In [156]:
import pickle
import os
import copy
# path = '/home/dan/mr_mpc_logs/2025-08-12_18:56:44_ur5e4O_bin1/stats.pkl' # './out/2025-08-09_11:55:02/stats.pkl'
# root_path = '/home/dan/Desktop/data_benchmarks/name_format_1/task_seed_0/selected_from_cluster'
root_path = '/home/dan/mr_mpc_logs/benchmarks_bin_test_aug17'
stat_file_name = 'stats.pkl'
simulation_dirs = os.listdir(root_path)

sim_names = []
sim_stats = []
for i, sim_dir in enumerate(simulation_dirs):
    print(f'Simulation {i+1} out of {len(simulation_dirs)}')
    print(f'Simulation dir: {sim_dir}')
    stats_path = os.path.join(root_path, sim_dir, stat_file_name)
    if os.path.exists(stats_path):
        with open(stats_path, 'rb') as f:
            stats = pickle.load(f)
            sim_names.append(sim_dir)
            sim_stats.append(stats)
        # print(stats)
    else:
        print(f'Stats file not found for simulation {sim_dir}')
        print('-'*100)





Simulation 1 out of 5
Simulation dir: 2025-08-17_16:52:17_R_ur5e_N4_ACC_Treach_s0_l3
Simulation 2 out of 5
Simulation dir: 2025-08-17_16:49:45_R_ur5e_N4_AO_Tfollow_s0_l3
Simulation 3 out of 5
Simulation dir: 2025-08-17_16:53:17_R_ur5e_N4_ACC_Tfollow_s0_l3
Simulation 4 out of 5
Simulation dir: 2025-08-17_16:46:15_R_ur5e_N4_AO_Treach_s0_l3
Simulation 5 out of 5
Simulation dir: 2025-08-17_16:41:33_R_ur5e_N4_AO_Tbin_s0_l3


In [157]:
print(f'stat manager names:')
sim_stats[0].keys()


stat manager names:


dict_keys(['task_stats', 'agent_0'])

In [158]:
def get_statman_keys(statman_data):
    return stats[statman_data].keys()

def get_stat_keys(statman_data, stat_name):
    return statman_data[stat_name].keys()

def get_stat_vals(statman_data, stat_name):
    return statman_data[stat_name]

def get_sim_keys(sim_name, sim_stats):
    for sm_name in sim_stats.keys(): # statman name
        for stat_name in sim_stats[sm_name].keys():
            print(f'simname: {sim_name}, sman: {sm_name}, s:{stat_name}')


Sim info

In [159]:

for sim_name, sim_stat in zip(sim_names, sim_stats):
    print(f'Sim info:')
    get_sim_keys(sim_name, sim_stat)
    print('')

    

Sim info:
simname: 2025-08-17_16:52:17_R_ur5e_N4_ACC_Treach_s0_l3, sman: task_stats, s:arm_err
simname: 2025-08-17_16:52:17_R_ur5e_N4_ACC_Treach_s0_l3, sman: task_stats, s:arm_changed
simname: 2025-08-17_16:52:17_R_ur5e_N4_ACC_Treach_s0_l3, sman: task_stats, s:arm_reached
simname: 2025-08-17_16:52:17_R_ur5e_N4_ACC_Treach_s0_l3, sman: agent_0, s:env_cols
simname: 2025-08-17_16:52:17_R_ur5e_N4_ACC_Treach_s0_l3, sman: agent_0, s:link_target_poses
simname: 2025-08-17_16:52:17_R_ur5e_N4_ACC_Treach_s0_l3, sman: agent_0, s:arm_cols
simname: 2025-08-17_16:52:17_R_ur5e_N4_ACC_Treach_s0_l3, sman: agent_0, s:total_planning_time
simname: 2025-08-17_16:52:17_R_ur5e_N4_ACC_Treach_s0_l3, sman: agent_0, s:spheres

Sim info:
simname: 2025-08-17_16:49:45_R_ur5e_N4_AO_Tfollow_s0_l3, sman: task_stats, s:arm_err
simname: 2025-08-17_16:49:45_R_ur5e_N4_AO_Tfollow_s0_l3, sman: agent_0, s:env_cols
simname: 2025-08-17_16:49:45_R_ur5e_N4_AO_Tfollow_s0_l3, sman: agent_0, s:link_target_poses
simname: 2025-08-17_16

In [160]:
# sim_stats[0]['task_stats'].keys()
# for stat_name in sim_stats[0]['task_stats'].keys():
#     print(f'stat name: {stat_name}')
#     print(sim_stats[0]['task_stats'][stat_name])
#     print('-'*100)

def find_task_statman_name(stats):
    for statman_name in stats.keys():
        if 'agent' not in statman_name:
            return statman_name
    return None

def find_agent_statman_names(stats):
    agent_statman_names = []
    for statman_name in stats.keys():
        if 'agent' in statman_name:
            agent_statman_names.append(statman_name)
    return agent_statman_names

def get_stat_from_statman(statman, stat_name, keys='all'):
    stat = statman[stat_name]
    if keys == 'all':
        return stat
    
    stat_filtered = []
    for stat_keys,stat_vals in stat.items():
        filtered_entry_keys = {}
        for key in keys:
            if key in stat_keys:
                filtered_entry_keys[key] = stat_vals[key]
                filtered_entry_vals = stat_vals
                stat_filtered.append((filtered_entry_keys,filtered_entry_vals))
    return stat_filtered
    
    
def split_task_to_k_v(stat):
    k = []
    v = []
    for item in stat:
        k.append(item[0])
        v.append(item[1])
    return k, v

def parse_task_by_dirname(root_path, simulation_dir):
    if 'name_format_1' in root_path:
        task_string = simulation_dir.split('_')[-1]
        task_type = task_string[:-1]
        task_seed = 0
        task_level = task_string[-1]
    
    # elif 'name_format_2' in root_path:
    else:
        task_string = simulation_dir.split('T')[-1]
        task_string_splitted = task_string.split('_')
        task_type = task_string_splitted[0]
        assert len(task_string_splitted) == 3, f'task string {task_string} is not in the correct format'
        assert task_string_splitted[1][0] == 's' and task_string_splitted[2][0] == 'l', f'task string {task_string} is not in the correct format'
        task_seed = int(task_string_splitted[1][1:])
        task_level = int(task_string_splitted[2][1:])
    # else:
    #     raise ValueError(f'root path {root_path} not supported')
    assert task_type in ['bin', 'reach','manual','follow'], f'task type {task_type} not supported'
    return task_type, task_seed, task_level

    

def pasrse_sim_info_by_dirname(root_path, simulation_dir):
    if 'name_format_1' in root_path:
        # example: 2025-08-16_16:43:16_ur5e4O_bin3 
        splitted_string = simulation_dir.split('_')
        timestamp = splitted_string[0] + '_' + splitted_string[1]
        robot_arms_alg = splitted_string[2]
        n_arms = 4
        robot_name, alg = robot_arms_alg.split('4')
        assert robot_name in ['ur5e','ur10e'], f'robot_name {robot_name} is not valid'
        task_parsed = parse_task_by_dirname(root_path, simulation_dir)
        task_type, task_seed, task_level = task_parsed
        
        
        # ans = timestamp, robot_name, n_arms, alg, task_type, task_seed, task_level
    else:  
    # elif 'name_format_2' in root_path:
        # example: # 2025-08-16_16:43:16_ur5e4O_bin3 
        splitted_string = simulation_dir.split('_')
        timestamp = splitted_string[0] + '_' + splitted_string[1]
        robot_name = splitted_string[3]
        n_arms = int(splitted_string[4][1:])
        alg = splitted_string[5][1:]
        task_parsed = parse_task_by_dirname(root_path, simulation_dir)
        task_type, task_seed, task_level = task_parsed

        # ans =  timestamp, robot_name, n_arms, alg, task_type, task_seed, task_level
    # else:
    #     raise ValueError(f'root path {root_path} not supported')
    
    return {'timestamp':timestamp, 'robot_name':robot_name, 'n_arms':n_arms, 'alg':alg, 'task_type':task_type, 'task_seed':task_seed, 'task_level':task_level}


class SimStats:
    def __init__(self, meta_data, sim_stats, sort_by=['w_step']):
        self.meta_data = meta_data
        self.sim_stats = sim_stats
        self.task_stats = sim_stats['task_stats']
        self.agent_stats = [sim_stats[x] for x in sim_stats.keys() if 'agent' in x]
        
        self.parsed = self.parse()
        if len(sort_by) > 0:
            self.parsed = sorted(self.parsed, key=lambda x: x[sort_by[0]])


    def parse(self):
        parsed = []
        print(f'parsing sim: {self}...')
        for statman_name in self.sim_stats.keys():
            print(f'\nparsing stat-manager: {statman_name}...')
            for stat_name in self.sim_stats[statman_name].keys():
                print(f'    parsing stat: {stat_name}...')
                stat_vals = self.sim_stats[statman_name][stat_name]
                # stat_entries = []
                for stat_kv in stat_vals:        
                    # stat entry (unique key)
                    s_entry = {}
                    s_entry[f'{statman_name}_{stat_name}'] = stat_kv[1]
                    for k_key, v_key in stat_kv[0].items():
                        s_entry[k_key] = v_key
                    # stat_entries.append(s_entry)
                    parsed.append(s_entry)
                # stat_entries.append(s_entry)
        print(f'parsing finished, parsed {len(parsed)} entries')
        return parsed
        
    def __repr__(self):
        out = '\n'
        for k,v in self.meta_data.items():
            out += f'{k}: {v}\n'
        return out
        
    def check_necessary_key(self, entry:dict,key:str):
        for k_entry in entry.keys():
            if key in k_entry:
                return True # found the key substring

        return False
    
class ReachSimStats(SimStats):
    
    def __init__(self, meta_data, sim_stats, sort_by=['w_step']):
        super().__init__(meta_data, sim_stats, sort_by)
        self.parse_task_stats()
        
    def parse_task_stats(self):
        self.n_arms = self.meta_data['n_arms']
        self.task_stats_parsed = []
        tmp_reached_by_t = [0 for _ in range(self.n_arms)]
        tmp_changed_by_t = [0 for _ in range(self.n_arms)]
        
        sorted_parsed = sorted(self.parsed, key=lambda x: x['w_step'])
        print(f'len(sorted_parsed): {len(sorted_parsed)}')
        for entry in sorted_parsed:
            
            if not self.check_necessary_key(entry,'task_stats_arm_reached'):
                continue # not a task related entry
            
            arm_stats_entry = {}
            entry_keys = entry.keys()

            
            
            found_required = False
            for required in ['task_stats_arm_reached', 'task_stats_arm_changed']:
                for k in entry_keys:
                    if required in k:
                        found_required = True
                        break
            if not found_required:
                print(f'{required} not found in {entry_keys}')
                continue
        
            for k in entry.keys():
                

                if k == 'task_stats_arm_reached':
                    for arm_idx in entry[k]:
                        tmp_reached_by_t[arm_idx] += 1
                
                if k == 'task_stats_arm_changed':
                    for arm_idx in entry[k]:
                        tmp_changed_by_t[arm_idx] += 1
                
            arm_stats_entry['w_step'] = entry['w_step']
            for arm_idx in range(self.n_arms):
                arm_stats_entry[f'arm_{arm_idx}_total_reached'] = tmp_reached_by_t[arm_idx]
            
            for arm_idx in range(self.n_arms):
                arm_stats_entry[f'arm_{arm_idx}_total_changed'] = tmp_changed_by_t[arm_idx]
        
            self.task_stats_parsed.append(arm_stats_entry)

    
class BinSimStats(SimStats):
    def __init__(self, meta_data, sim_stats, sort_by=['w_step']):
        super().__init__(meta_data, sim_stats, sort_by)
        self.parse_task_stats()


    def parse_task_stats(self):
        self.n_arms = self.meta_data['n_arms']
        self.task_stats_parsed = []
        
        tmp_picks_by_t = [0 for _ in range(self.n_arms)]
        tmp_drops_by_t = [0 for _ in range(self.n_arms)]
        
        sorted_parsed = sorted(self.parsed, key=lambda x: x['w_step'])
        for entry in sorted_parsed:
            arm_stats_entry = {}
            
            if not self.check_necessary_key(entry,'arm_picks'):
                continue # not a task related entry
            
            for k in entry.keys():
                if k in ['w_step']:
                    arm_stats_entry[k] = entry[k]
                    
                if 'arm_err' in k:
                    for arm_idx in entry[k]:
                        arm_stats_entry[f'arm_{arm_idx}_err'] = entry[k][arm_idx]
                if 'arm_picks' in k:
                    for arm_idx in entry[k]:
                        tmp_picks_by_t[arm_idx] += 1
                        arm_stats_entry[f'arm_{arm_idx}_total_picks'] = tmp_picks_by_t[arm_idx]
                if 'arm_drops' in k:
                    for arm_idx in entry[k]:
                        tmp_drops_by_t[arm_idx] += 1
                        arm_stats_entry[f'arm_{arm_idx}_total_drops'] = tmp_drops_by_t[arm_idx]
                        
            self.task_stats_parsed.append(arm_stats_entry)
                    
class FollowSimStats(SimStats):
    def __init__(self, meta_data, sim_stats, sort_by=['w_step']):
        super().__init__(meta_data, sim_stats, sort_by)
        self.parse_task_stats()
        
    def parse_task_stats(self):
        self.n_arms = self.meta_data['n_arms']
        self.task_stats_parsed = []
        
        sorted_parsed = sorted(self.parsed, key=lambda x: x['w_step'])
        for entry in sorted_parsed:
            arm_stats_entry = {}
            
            if not self.check_necessary_key(entry,'arm_err'):
                continue # not a task related entry
                
            for k in entry.keys():
                if k in ['w_step']:
                    arm_stats_entry[k] = entry[k]
                        
            self.task_stats_parsed.append(arm_stats_entry)






# Parse all sims


In [161]:
reach_sims = []
bin_sims = []
follow_sims = []

for i, this_sim_stats in enumerate(sim_stats):
    meta_data = pasrse_sim_info_by_dirname(root_path, sim_names[i])
    print(meta_data["task_type"])
    if meta_data['task_type'] == 'reach':
        reach_sims.append(ReachSimStats(meta_data, this_sim_stats))
    # elif meta_data['task_type'] == 'bin':
    #     bin_sims.append(BinSimStats(meta_data, this_sim_stats))
    elif meta_data['task_type'] == 'follow':
        follow_sims.append(FollowSimStats(meta_data, this_sim_stats))
    # else:
    #     raise ValueError(f'task type {meta_data["task_type"]} not supported')
    print('-'*100)




reach
parsing sim: 
timestamp: 2025-08-17_16:52:17
robot_name: ur5e
n_arms: 4
alg: CC
task_type: reach
task_seed: 0
task_level: 3
...

parsing stat-manager: task_stats...
    parsing stat: arm_err...
    parsing stat: arm_changed...
    parsing stat: arm_reached...

parsing stat-manager: agent_0...
    parsing stat: env_cols...
    parsing stat: link_target_poses...
    parsing stat: arm_cols...
    parsing stat: total_planning_time...
    parsing stat: spheres...
parsing finished, parsed 3559 entries
len(sorted_parsed): 3559
----------------------------------------------------------------------------------------------------
follow
parsing sim: 
timestamp: 2025-08-17_16:49:45
robot_name: ur5e
n_arms: 4
alg: O
task_type: follow
task_seed: 0
task_level: 3
...

parsing stat-manager: task_stats...
    parsing stat: arm_err...

parsing stat-manager: agent_0...
    parsing stat: env_cols...
    parsing stat: link_target_poses...
    parsing stat: arm_cols...
    parsing stat: total_planning_

In [162]:
for reached_sim in reach_sims:
    for i, entry in enumerate(reached_sim.task_stats_parsed):
        print(f'{i}: {entry}')

        


0: {'w_step': 1, 'arm_0_total_reached': 0, 'arm_1_total_reached': 0, 'arm_2_total_reached': 0, 'arm_3_total_reached': 0, 'arm_0_total_changed': 0, 'arm_1_total_changed': 0, 'arm_2_total_changed': 0, 'arm_3_total_changed': 0}
1: {'w_step': 2, 'arm_0_total_reached': 0, 'arm_1_total_reached': 0, 'arm_2_total_reached': 0, 'arm_3_total_reached': 0, 'arm_0_total_changed': 0, 'arm_1_total_changed': 0, 'arm_2_total_changed': 0, 'arm_3_total_changed': 0}
2: {'w_step': 3, 'arm_0_total_reached': 0, 'arm_1_total_reached': 0, 'arm_2_total_reached': 0, 'arm_3_total_reached': 0, 'arm_0_total_changed': 0, 'arm_1_total_changed': 0, 'arm_2_total_changed': 0, 'arm_3_total_changed': 0}
3: {'w_step': 4, 'arm_0_total_reached': 0, 'arm_1_total_reached': 0, 'arm_2_total_reached': 0, 'arm_3_total_reached': 0, 'arm_0_total_changed': 0, 'arm_1_total_changed': 0, 'arm_2_total_changed': 0, 'arm_3_total_changed': 0}
4: {'w_step': 5, 'arm_0_total_reached': 0, 'arm_1_total_reached': 0, 'arm_2_total_reached': 0, 'arm_

In [167]:
reach_sims[1].task_stats_parsed
reach_sims[1]


timestamp: 2025-08-17_16:46:15
robot_name: ur5e
n_arms: 4
alg: O
task_type: reach
task_seed: 0
task_level: 3

In [164]:
# 'arm_reached' in reach_sims[0].parsed[6].keys()

sorted_parsed = sorted(reach_sims[0].parsed, key=lambda x: x['w_step'])
print(f'len(sorted_parsed): {len(sorted_parsed)}')
for entry in sorted_parsed:
    
    print(entry.keys())
    if 'task_stats_arm_reached' in entry.keys():
        print(True)
 

len(sorted_parsed): 3559
dict_keys(['agent_0_env_cols', 'tsys', 'tphysics', 'w_step', 'a_step'])
dict_keys(['agent_0_link_target_poses', 'tsys', 'tphysics', 'w_step', 'a_step'])
dict_keys(['agent_0_arm_cols', 'tsys', 'tphysics', 'w_step', 'a_step'])
dict_keys(['agent_0_total_planning_time', 'tsys', 'tphysics', 'w_step', 'a_step'])
dict_keys(['task_stats_arm_err', 'tsys', 'tphysics', 'w_step'])
dict_keys(['task_stats_arm_changed', 'tsys', 'tphysics', 'w_step'])
dict_keys(['task_stats_arm_reached', 'tsys', 'tphysics', 'w_step'])
True
dict_keys(['agent_0_env_cols', 'tsys', 'tphysics', 'w_step', 'a_step'])
dict_keys(['agent_0_link_target_poses', 'tsys', 'tphysics', 'w_step', 'a_step'])
dict_keys(['agent_0_arm_cols', 'tsys', 'tphysics', 'w_step', 'a_step'])
dict_keys(['agent_0_total_planning_time', 'tsys', 'tphysics', 'w_step', 'a_step'])
dict_keys(['task_stats_arm_err', 'tsys', 'tphysics', 'w_step'])
dict_keys(['task_stats_arm_changed', 'tsys', 'tphysics', 'w_step'])
dict_keys(['task_stats